In [5]:
import re
import sys

USE_MAH_FOR_SINGLE = True


CONJUNCTS = {
    "क्ष": "ksh",  "ज्ञ": "gya",  "श्र": "shr",
    "प्र": "pra",  "त्र": "tra",  "स्त": "sta",
    "स्थ": "stha", "स्व": "swa",  "द्व": "dwa",
    "द्य": "dya",  "ह्य": "hya",  "ह्र": "hra",
    "ह्व": "hwa",  "न्ह": "nha",  "म्ह": "mha",
    "ल्ह": "lha",  "न्न": "nna",  "त्त": "tta",
    "क्क": "kka",  "ग्ग": "gga",  "ब्ब": "bba",
    "म्म": "mma",  "न्द": "nda",  "न्त": "nta",
    "ङ्क": "ngka", "ङ्ग": "ngga", "ञ्च": "ncha",
    "ञ्ज": "nja",
}       

CONSONANTS = {
    "क": "k",  "ख": "kh", "ग": "g",  "घ": "gh", "ङ": "ng",
    "च": "ch", "छ": "chh","ज": "j",  "झ": "jh", "ञ": "n",
    "ट": "t",  "ठ": "th", "ड": "d",  "ढ": "dh", "ण": "n",
    "त": "t",  "थ": "th", "द": "d",  "ध": "dh", "न": "n",
    "प": "p",  "फ": "ph", "ब": "b",  "भ": "bh", "म": "m",
    "य": "y",  "र": "r",  "ल": "l",
    "व": "w",
    "श": "sh", "ष": "sh", "स": "s",  "ह": "h",
}

VOWEL_SIGNS = {
    "ा": "a",  "ि": "i",  "ी": "i",  "ु": "u",  "ू": "u",
    "ृ": "ri", "े": "e",  "ै": "ai", "ो": "o",  "ौ": "au",
    "ं": "n",  "ँ": "",   "ः": "h",
    "\u094d": "",
}

INDEPENDENT_VOWELS = {
    "अ": "a",  "आ": "aa", "इ": "i",  "ई": "ii",
    "उ": "u",  "ऊ": "uu", "ऋ": "ri",
    "ए": "e",  "ऐ": "ai", "ओ": "o",  "औ": "au",
    "ऍ": "e",  "ऑ": "o",
}

POSTPOSITION_SPLITS = [
    (r'([a-z]{3,})lai\b',   r'\1 lai'),
    (r'([a-z]{3,})bata\b',  r'\1 bata'),
    (r'([a-z]{3,})sanga\b', r'\1 sanga'),
    (r'([a-z]{5,})ma\b',    r'\1 ma'),
    (r'([a-z]{6,})ko\b',    r'\1 ko'),
]

LOANWORDS = {
    "आर्सनललाई":    "Arsenal lai",
    "आर्सनल":       "Arsenal",
    "एथ्लेटिको":    "Atletico",
    "मड्रिड":       "Madrid",
    "युरोपा":       "Europa",
    "प्रिमियर":     "Premier",
    "लिगको":        "League ko",
    "लिग":          "League",
    "फाइनलमा":      "final ma",
    "फाइनल":        "final",
    "सिजन":         "season",
    "कप":           "Cup",
    "बेंगर":        "Wenger",
    "इमिरेट्स":    "Emirates",
    "रियल":         "Real",
    "फुटबल":        "football",
    "च्याम्पियन्स": "Champions",
    "काठमाडौं":     "Kathmandu",
    "वेबसाइट":       "Website",
    'फाइनलमा': 'final ma',
    'सिजनमा': 'season ma',
    'कपको': 'cup ko',
    'प्रशिक्षकका': 'coach ka',
    'प्रशिक्षकले': 'coach le',
    'बेंगरले': 'Wenger le',
    'इमिरेट्स छोड्ने': 'Emirates chodne',
    'उनले': 'unlay',          # human typing style
    'तीन वटा': 'tin baata',
    'सात वटा': 'sat baata',
    'मात्र': 'matra',          # keep final 'a'
}

DIGITS = {"०": "0", "१": "1", "२": "2", "३": "3", "४": "4",
          "५": "5", "६": "6", "७": "7", "८": "8", "९": "9"}

HALANT        = "\u094d"
PUNCT_MAP     = {"।": ".", "॥": ".", "–": "-", "—": "-", "…": "..."}
WORD_BOUNDARY = set(" ।॥\n\t?!,.-–—)]}")

# ═══════════════════════════════════════════════════════════════════════════
#  CORE TRANSLITERATOR (unchanged)
# ═══════════════════════════════════════════════════════════════════════════

def _is_consonant(c: str) -> bool:         return c in CONSONANTS
def _is_vowel_sign(c: str) -> bool:        return c in VOWEL_SIGNS
def _is_independent_vowel(c: str) -> bool: return c in INDEPENDENT_VOWELS
def _is_word_boundary(c: str) -> bool:     return c in WORD_BOUNDARY

def transliterate(text: str) -> str:
    result = []
    chars = list(text)
    n = len(chars)
    i = 0

    while i < n:
        # Loanword
        matched_loan = None
        for loan_len in range(min(10, n - i), 0, -1):
            chunk = "".join(chars[i:i + loan_len])
            if chunk in LOANWORDS:
                matched_loan = (chunk, loan_len)
                break
        if matched_loan:
            result.append(LOANWORDS[matched_loan[0]])
            i += matched_loan[1]
            continue

        ch = chars[i]

        if ch in DIGITS:
            result.append(DIGITS[ch])
            i += 1
            continue

        if ch in PUNCT_MAP:
            result.append(PUNCT_MAP[ch])
            i += 1
            continue

        if _is_independent_vowel(ch):
            result.append(INDEPENDENT_VOWELS[ch])
            i += 1
            continue

        # Conjunct
        conjunct_found = False
        for clen in (3, 2):
            chunk = "".join(chars[i:i + clen])
            if chunk in CONJUNCTS:
                next_i = i + clen
                rom = CONJUNCTS[chunk]
                if next_i < n and _is_vowel_sign(chars[next_i]):
                    vsign = chars[next_i]
                    result.append(rom if vsign == HALANT else rom + VOWEL_SIGNS[vsign])
                    i = next_i + 1
                else:
                    is_final = (next_i >= n or _is_word_boundary(chars[next_i]))
                    result.append(rom if is_final else rom + "a")
                    i = next_i
                conjunct_found = True
                break
        if conjunct_found:
            continue

        # Regular consonant
        if _is_consonant(ch):
            rom = CONSONANTS[ch]
            next_i = i + 1
            if next_i < n:
                nch = chars[next_i]
                if nch == HALANT:
                    result.append(rom)
                    i = next_i + 1
                    continue
                if _is_vowel_sign(nch):
                    result.append(rom + VOWEL_SIGNS[nch])
                    i = next_i + 1
                    continue
                if (_is_word_boundary(nch) or _is_consonant(nch) or _is_independent_vowel(nch)):
                    result.append(rom if _is_word_boundary(nch) else rom + "a")
                    i += 1
                    continue
            # End of string or no special case
            result.append(rom)   # no inherent 'a' at word end
            i += 1
            continue

        # Standalone vowel sign
        if _is_vowel_sign(ch):
            if ch != HALANT:
                result.append(VOWEL_SIGNS[ch])
            i += 1
            continue

        # Passthrough
        result.append(ch)
        i += 1

    return "".join(result)

# ═══════════════════════════════════════════════════════════════════════════
#  POST‑PROCESSING (with configurable single‑word forms)
# ═══════════════════════════════════════════════════════════════════════════

def _fix_va_word_initial(text: str) -> str:
    text = re.sub(r'^w', 'b', text)
    text = re.sub(r'(?<=[ \-\u2013\u2014,.(!\?])w', 'b', text)
    return text

def _fix_chha(text: str) -> str:
    return re.sub(r'chh(?=[^a-zA-Z]|$)', 'chha', text)

def _fix_anusvara_m(text: str) -> str:
    return re.sub(r'n([pbm])', lambda m: 'm' + m.group(1), text)

def _fix_tapain(text: str) -> str:
    text = re.sub(r'tapaii+n?laii+', 'tapai lai', text, flags=re.IGNORECASE)
    text = re.sub(r'tapai+n?lai+',   'tapai lai', text, flags=re.IGNORECASE)
    text = re.sub(r'tapaii+n?',      'tapai',     text, flags=re.IGNORECASE)
    return text

def _fix_single_char_words(text: str) -> str:
    """
    Standalone consonants (म, न, त, etc.) -> 'ma'/'mah', 'na'/'nah', etc.
    Controlled by USE_MAH_FOR_SINGLE flag.
    """
    if USE_MAH_FOR_SINGLE:
        singles = {'m': 'mah', 'n': 'nah', 't': 'tah', 'k': 'kah', 'r': 'rah'}
    else:
        singles = {'m': 'ma', 'n': 'na', 't': 'ta', 'k': 'ka', 'r': 'ra'}
    def repl(m): return singles.get(m.group(1), m.group(1))
    return re.sub(r'(?<![a-zA-Z])([mntkr])(?![a-zA-Z])', repl, text)

def _fix_aan_chandrabindu(text: str) -> str:
    return re.sub(r'\bja+n?(dai|nu|ne|chhu)', r'jaan\1', text)

def _fix_namaste(text: str) -> str:
    return re.sub(r'stae\b', 'ste', text)

def _fix_malai(text: str) -> str:
    return re.sub(r'\bmalaii\b', 'malai', text, flags=re.IGNORECASE)

def _fix_double_vowels(text: str) -> str:
    text = re.sub(r'([^aeiou])aa([^aeiou])', r'\1a\2', text)
    text = re.sub(r'([^aeiou])ao\b',         r'\1o',   text)
    return text

def _split_postpositions(text: str) -> str:
    for pattern, repl in POSTPOSITION_SPLITS:
        text = re.sub(pattern, repl, text)
    return text

def _clean_spaces(text: str) -> str:
    text = re.sub(r' +', ' ', text)
    text = re.sub(r' ([.,!?])', r'\1', text)
    return text.strip()

def _capitalize_first(text: str) -> str:
    return (text[0].upper() + text[1:]) if text else text

def _fix_haraau(text: str) -> str:
    return re.sub(r'\bharau', 'haraau', text)

def _post_process(text: str) -> str:
    text = _fix_va_word_initial(text)
    text = _fix_anusvara_m(text)
    text = _fix_chha(text)
    text = _fix_tapain(text)
    text = _fix_haraau(text)
    text = _fix_aan_chandrabindu(text)
    text = _fix_namaste(text)
    text = _fix_malai(text)
    text = _fix_double_vowels(text)
    text = _split_postpositions(text)
    text = _fix_single_char_words(text)   # uses the flag
    text = _clean_spaces(text)
    text = _capitalize_first(text)
    return text

def romanize_nepali(text: str) -> str:
    return _post_process(transliterate(text))


def main():
    input_file = "devnagari.txt"
    output_file = "output.txt"

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found.", file=sys.stderr)
        sys.exit(1)

    results = [romanize_nepali(line) if line.strip() else '' for line in lines]

    with open(output_file, 'w', encoding='utf-8') as out:
        for res in results:
            out.write(res + '\n')

    print(f"Romanized text written to '{output_file}'.")

if __name__ == "__main__":
    main()
    

Romanized text written to 'output.txt'.


In [6]:
import sys
import re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class NepaliRomanizerOllamaOnly:
    def __init__(self, model_name: str = "mistral-small3.1:latest", temperature: float = 0.0):
        self.model_name = model_name
        self.temperature = temperature
        self.llm = None
        self.chain = None
        self._initialize_model()
    
    def _initialize_model(self):
        """Initialize the Ollama model"""
        try:
            self.llm = ChatOllama(
                model=self.model_name,
                temperature=self.temperature,
                num_predict=1024,
            )
            
            # Optimized prompt for Mistral
            self.prompt = ChatPromptTemplate.from_messages([
                ("system", """You are a Nepali transliteration expert. Convert Devanagari Nepali text to Roman script.

                        CRITICAL RULES:
                        - Output ONLY the romanized text. NO greetings, NO "Sure", NO "Here is", NO explanations.
                        - Start the output IMMEDIATELY with the converted text.
                        - Use ONLY plain ASCII letters (a-z). NO diacritics.
                        - Preserve punctuation, spaces, and line breaks exactly.

                        Romanization examples:
                        यसबाट → yasbata
                        मलाई → malai
                        दुख → dukh
                        छैन → chaina
                        राष्ट्र्प्रेमी → rashtrapremi
                        साथीहरुले → sathiharle
                        मान्नु → mannu
                        धन्यवाद → dhanyabad
                        भविष्यमा → bhavishyama
                        गरेको → gareko
                        रहेछौं → rahechhau

                        Output format: Start directly with the converted text, no prefixes."""),
                                        ("human", "{devanagari_text}")
                ])
            
            self.chain = self.prompt | self.llm | StrOutputParser()
            print(f"✓ Model '{self.model_name}' loaded successfully", file=sys.stderr)
            
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            print("  Make sure Ollama is running: 'ollama serve'", file=sys.stderr)
            print(f"  And model is pulled: 'ollama pull {self.model_name}'", file=sys.stderr)
            sys.exit(1)
    
    def _clean_output(self, text: str) -> str:
        """Remove unwanted prefixes and clean output"""
        # Remove common prefixes
        prefixes = [
            "Sure, here is the Romanized version:",
            "Here is the Romanized version:",
            "Here is the romanized text:",
            "Romanized text:",
            "Output:",
            "Result:",
        ]
        
        for prefix in prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()
        
        # Remove quotes
        text = re.sub(r'^["\']|["\']$', '', text)
        
        # Ensure proper spacing
        text = re.sub(r'\s+', ' ', text)
        
        # Capitalize first letter of first sentence
        if text and text[0].islower():
            text = text[0].upper() + text[1:]
        
        return text.strip()
    
    def romanize(self, text: str) -> str:
        """Convert Devanagari Nepali text to Roman script"""
        if not text or not text.strip():
            return ""
        
        try:
            # Clean input text
            text = text.strip()
            
            # Invoke the model
            result = self.chain.invoke({"devanagari_text": text})
            
            # Clean up the result
            result = result.strip()
            result = self._clean_output(result)
            
            return result
            
        except Exception as e:
            print(f"✗ Error converting text: {e}", file=sys.stderr)
            print(f"  Text: {text[:50]}...", file=sys.stderr)
            return text
    
    def romanize_batch(self, texts: list) -> list:
        """Convert multiple texts to Roman script"""
        results = []
        total = len(texts)
        
        for idx, text in enumerate(texts, 1):
            if idx % 10 == 0:
                print(f"  Processing {idx}/{total}...", file=sys.stderr)
            
            result = self.romanize(text)
            results.append(result)
        
        return results


def main():
    input_file = "devnagari.txt"
    output_file = "output.txt"
    
    # Use Mistral Small 3.1 - best quality for Nepali
    MODEL_NAME = "mistral-small3.1:latest"
    TEMPERATURE = 0.0
    
    print("=" * 60, file=sys.stderr)
    print("NEPALI ROMANIZATION WITH MISTRAL SMALL 3.1", file=sys.stderr)
    print("=" * 60, file=sys.stderr)
    print(f"Model: {MODEL_NAME}", file=sys.stderr)
    print(f"Temperature: {TEMPERATURE}", file=sys.stderr)
    print("", file=sys.stderr)
    
    # Initialize the romanizer
    romanizer = NepaliRomanizerOllamaOnly(
        model_name=MODEL_NAME,
        temperature=TEMPERATURE
    )
    
    # Read input file
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]
        print(f"✓ Read {len(lines)} lines from '{input_file}'", file=sys.stderr)
        print("", file=sys.stderr)
    except FileNotFoundError:
        print(f"✗ Error: Input file '{input_file}' not found.", file=sys.stderr)
        sys.exit(1)
    
    # Process lines
    print("Converting to Roman script...", file=sys.stderr)
    results = romanizer.romanize_batch(lines)
    
    # Write output file
    with open(output_file, 'w', encoding='utf-8') as out:
        for res in results:
            out.write(res + '\n')
    
    print("", file=sys.stderr)
    print("=" * 60, file=sys.stderr)
    print(f"✓ Conversion complete!", file=sys.stderr)
    print(f"  Output written to: '{output_file}'", file=sys.stderr)
    print("=" * 60, file=sys.stderr)


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

In [ ]:
import sys
import re
import time
import json
from pathlib import Path
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class NepaliRomanizerBatch:
    def __init__(self, model_name: str = "mistral-small3.1:latest", temperature: float = 0.0, batch_size: int = 5):
        self.model_name = model_name
        self.temperature = temperature
        self.batch_size = batch_size
        self.llm = None
        self.chain = None
        self._initialize_model()
    
    def _initialize_model(self):
        """Initialize the Ollama model"""
        try:
            self.llm = ChatOllama(
                model=self.model_name,
                temperature=self.temperature,
                num_predict=2048,  # Increased for batch
                num_ctx=4096,      # Larger context for batches
            )
            
            # Optimized prompt for batch processing
            self.prompt = ChatPromptTemplate.from_messages([
                ("system", """You are a Nepali transliteration expert. Convert Devanagari Nepali text to Roman script.

CRITICAL RULES:
- Output ONLY the romanized text for EACH line
- Keep the SAME ORDER as input
- Separate each output line with exactly: [SEP]
- NO explanations, NO "Sure", NO "Here is"
- Use plain ASCII, no diacritics

Example:
Input: नेपाल
राष्ट्र बैंक
Output: Nepal[SEP]Rashtra Bank

Follow this format exactly."""),
                ("human", "Convert these {count} Nepali lines to Roman script. Output each line separated by [SEP]:\n\n{lines}")
            ])
            
            self.chain = self.prompt | self.llm | StrOutputParser()
            print(f"✓ Model '{self.model_name}' loaded successfully", file=sys.stderr)
            print(f"✓ Batch size: {self.batch_size} lines per request", file=sys.stderr)
            
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            sys.exit(1)
    
    def _clean_output(self, text: str) -> str:
        """Clean model output"""
        prefixes = [
            "Sure, here is the Romanized version:",
            "Here is the Romanized text:",
            "Romanized text:",
            "Output:",
            "Result:",
        ]
        for prefix in prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()
        return text.strip()
    
    def romanize_batch(self, lines: list) -> list:
        """Convert multiple lines in one API call"""
        if not lines:
            return []
        
        try:
            combined_lines = "\n".join(lines)
            result = self.chain.invoke({
                "count": len(lines),
                "lines": combined_lines
            })
            
            result = self._clean_output(result)
            romanized_lines = [line.strip() for line in result.split("[SEP]")]
            romanized_lines = [line for line in romanized_lines if line]
            
            # If we got wrong number, fallback to individual
            if len(romanized_lines) != len(lines):
                print(f"  Warning: Expected {len(lines)} lines, got {len(romanized_lines)}", file=sys.stderr)
                return [self.romanize_single(line) for line in lines]
            
            return romanized_lines
            
        except Exception as e:
            print(f"  Batch error: {e}", file=sys.stderr)
            return [self.romanize_single(line) for line in lines]
    
    def romanize_single(self, text: str) -> str:
        """Fallback: Convert single line"""
        try:
            single_prompt = ChatPromptTemplate.from_messages([
                ("system", "Convert Devanagari Nepali to Roman script. Output ONLY the romanized text."),
                ("human", "{text}")
            ])
            single_chain = single_prompt | self.llm | StrOutputParser()
            result = single_chain.invoke({"text": text})
            return result.strip()
        except Exception as e:
            print(f"  Error: {e}", file=sys.stderr)
            return text
    
    def process_file(self, input_file: str, output_file: str, max_lines: int = None, resume: bool = True):
        """Process large file with batch optimization"""
        
        start_time = time.time()
        
        # Read input file
        print(f"\n📖 Reading {input_file}...", file=sys.stderr)
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]
        
        if max_lines:
            lines = lines[:max_lines]
        
        total = len(lines)
        print(f"✓ Total lines: {total:,}", file=sys.stderr)
        
        # Check for existing progress
        processed_lines = 0
        results = []
        
        if resume and Path(output_file).exists():
            with open(output_file, 'r', encoding='utf-8') as f:
                results = [line.rstrip('\n') for line in f]
                processed_lines = len(results)
                print(f"✓ Resuming from line {processed_lines:,}", file=sys.stderr)
                
                # Remove already processed lines
                lines = lines[processed_lines:]
        
        # Calculate batches
        num_batches = (len(lines) + self.batch_size - 1) // self.batch_size
        print(f"✓ Batches to process: {num_batches:,}", file=sys.stderr)
        
        # Estimate time
        if num_batches > 0:
            estimated_seconds = num_batches * 3  # ~3 seconds per batch
            print(f"✓ Estimated time: {self._format_time(estimated_seconds)}", file=sys.stderr)
        
        # Process batches with tqdm
        print(f"\n🚀 Starting batch processing...\n", file=sys.stderr)
        
        with tqdm(total=len(lines), desc="Romanizing", unit="lines", 
                  bar_format='{l_bar}{bar:40}{r_bar}{bar:-10b}') as pbar:
            
            for i in range(0, len(lines), self.batch_size):
                batch = lines[i:i+self.batch_size]
                batch_start = time.time()
                
                # Process batch
                romanized_batch = self.romanize_batch(batch)
                results.extend(romanized_batch)
                
                # Update progress
                batch_time = time.time() - batch_start
                pbar.update(len(batch))
                
                # Update progress bar description
                pbar.set_postfix({
                    'batch': f"{i//self.batch_size + 1}/{num_batches}",
                    'time': f"{batch_time:.1f}s"
                })
                
                # Save checkpoint every 50 batches
                if (i // self.batch_size) % 50 == 0 and i > 0:
                    self._save_results(results, output_file)
                    self._save_checkpoint(processed_lines + i + len(batch))
                    print(f"\n  💾 Checkpoint saved at line {processed_lines + i + len(batch):,}", file=sys.stderr)
        
        # Final save
        self._save_results(results, output_file)
        
        elapsed = time.time() - start_time
        print(f"\n{'='*60}", file=sys.stderr)
        print(f"✅ CONVERSION COMPLETE!", file=sys.stderr)
        print(f"{'='*60}", file=sys.stderr)
        print(f"Total lines: {total:,}", file=sys.stderr)
        print(f"Total time: {self._format_time(elapsed)}", file=sys.stderr)
        print(f"Speed: {total/elapsed:.2f} lines/second", file=sys.stderr)
        print(f"Output: {output_file}", file=sys.stderr)
        print(f"{'='*60}", file=sys.stderr)
    
    def _save_results(self, results: list, output_file: str):
        """Save results to file"""
        with open(output_file, 'w', encoding='utf-8') as f:
            for result in results:
                f.write(result + '\n')
    
    def _save_checkpoint(self, line_number: int):
        """Save checkpoint"""
        checkpoint = {
            'line_number': line_number,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        with open('checkpoint.json', 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, indent=2)
    
    def _format_time(self, seconds: float) -> str:
        """Format time"""
        if seconds < 60:
            return f"{seconds:.0f} seconds"
        elif seconds < 3600:
            return f"{seconds/60:.1f} minutes"
        elif seconds < 86400:
            return f"{seconds/3600:.1f} hours"
        else:
            return f"{seconds/86400:.1f} days"


def main():
    input_file = "ne.txt"
    output_file = "ne_romanized.txt"
    
    print("=" * 70, file=sys.stderr)
    print("NEPALI ROMANIZATION - BATCH PROCESSOR", file=sys.stderr)
    print("=" * 70, file=sys.stderr)
    
    # Check input file
    if not Path(input_file).exists():
        print(f"✗ Error: Input file '{input_file}' not found.", file=sys.stderr)
        sys.exit(1)
    
    file_size_mb = Path(input_file).stat().st_size / (1024 * 1024)
    print(f"✓ Input file: {input_file} ({file_size_mb:.1f} MB)", file=sys.stderr)
    
    # Configuration
    MODEL_NAME = "mistral-small3.1:latest"
    BATCH_SIZE = 5  # Smaller batch for stability
    TEMPERATURE = 0.0
    
    print(f"✓ Model: {MODEL_NAME}", file=sys.stderr)
    print(f"✓ Batch size: {BATCH_SIZE} lines/request", file=sys.stderr)
    
    # Ask user
    response = input("\nStart processing? (y/n): ").strip().lower()
    if response != 'y':
        print("Aborted.", file=sys.stderr)
        sys.exit(0)
    
    # Initialize and process
    romanizer = NepaliRomanizerBatch(
        model_name=MODEL_NAME,
        temperature=TEMPERATURE,
        batch_size=BATCH_SIZE
    )
    
    # Process with option to test first 100 lines
    test_first = input("\nTest with first 100 lines first? (y/n): ").strip().lower()
    if test_first == 'y':
        print("\n🧪 Testing with first 100 lines...", file=sys.stderr)
        romanizer.process_file(input_file, "test_output.txt", max_lines=100)
        print("\n✓ Test complete. Check 'test_output.txt'", file=sys.stderr)
        
        proceed = input("\nProcess full file? (y/n): ").strip().lower()
        if proceed != 'y':
            print("Aborted.", file=sys.stderr)
            sys.exit(0)
    
    # Process full file
    romanizer.process_file(input_file, output_file)


if __name__ == "__main__":
    # Install tqdm if needed
    try:
        from tqdm import tqdm
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
        from tqdm import tqdm
    
    main()0.

NEPALI ROMANIZATION - STREAMING BATCH PROCESSOR
2026-04-10 23:03:50,276 - INFO - Model: mistral-small3.1:latest
2026-04-10 23:03:50,277 - INFO - Batch size: 5 lines per request
2026-04-10 23:03:50,277 - INFO - Input file size: 3851.7 MB
2026-04-10 23:03:50,278 - INFO - Loading model: mistral-small3.1:latest
2026-04-10 23:03:50,286 - INFO - ✓ Model 'mistral-small3.1:latest' loaded successfully
2026-04-10 23:03:50,286 - INFO - ✓ Batch size: 5 lines per request
2026-04-10 23:03:50,287 - INFO - 
⚠️  Processing will take ~44 hours
2026-04-10 23:03:50,287 - INFO - ⚠️  Progress saved every 1000 lines - Ctrl+C to interrupt

2026-04-10 23:03:50,287 - INFO - Counting total lines...
2026-04-10 23:03:57,929 - INFO - ✓ Total lines: 12,732,810
2026-04-10 23:03:57,930 - INFO - ✓ Resuming from line 5,000


Romanizing:   0%|                                        | 0/12727810 [00:00<?, ?lines/s]

2026-04-10 23:04:13,234 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-10 23:06:15,690 - WARNING - Expected 5 lines, got 7. Using fallback.
2026-04-10 23:06:18,818 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-10 23:06:21,910 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-10 23:06:35,812 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-10 23:06:56,170 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-10 23:07:44,942 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Romanizing:   0%|                                        | 1/12727810 [04:52<1035355:44:18, 292.85s/lines]

2026-04-10 23:08:53,169 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Romanizing:   0%|                                        | 6/12727810 [05:47<161183:32:52, 45.59s/lines]  

2026-04-10 23:09:47,806 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [ ]:
import sys
import re
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class NepaliRomanizerLocal:
    def __init__(self, model_path: str = "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf", temperature: float = 0.0):
        self.model_path = model_path
        self.temperature = temperature
        self.llm = None
        self.chain = None
        self._initialize_model()
    
    def _initialize_model(self):
        """Initialize the local Mistral model using llama-cpp-python"""
        try:
            self.llm = LlamaCpp(
                model_path=self.model_path,
                temperature=self.temperature,
                max_tokens=1024,
                n_ctx=2048,  # Context window
                n_gpu_layers=-1,  # Use GPU if available (-1 = all layers)
                n_batch=512,  # Batch size for prompt processing
                verbose=False,
                # For CPU-only or to control threads:
                n_threads=None,  # Use all available threads
                # Stop sequences to prevent unwanted continuation
                stop=["Human:", "User:", "\n\n\n"],
            )
            
            # Optimized prompt for Mistral 7B
            self.prompt = ChatPromptTemplate.from_messages([
                ("system", """You are a Nepali transliteration expert. Convert Devanagari Nepali text to Roman script.

CRITICAL RULES:
- Output ONLY the romanized text. NO greetings, NO "Sure", NO "Here is", NO explanations.
- Start the output IMMEDIATELY with the converted text.
- Use ONLY plain ASCII letters (a-z). NO diacritics.
- Preserve punctuation, spaces, and line breaks exactly.

Romanization examples:
यसबाट → yasbata
मलाई → malai
दुख → dukh
छैन → chaina
राष्ट्र्प्रेमी → rashtrapremi
साथीहरुले → sathiharle
मान्नु → mannu
धन्यवाद → dhanyabad
भविष्यमा → bhavishyama
गरेको → gareko
रहेछौं → rahechhau

Output format: Start directly with the converted text, no prefixes."""),
                ("human", "{devanagari_text}")
            ])
            
            self.chain = self.prompt | self.llm | StrOutputParser()
            print(f"✓ Model loaded successfully from '{self.model_path}'", file=sys.stderr)
            
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            print("  Make sure you have installed llama-cpp-python:", file=sys.stderr)
            print("  pip install llama-cpp-python", file=sys.stderr)
            print(f"  And the model file exists at: {self.model_path}", file=sys.stderr)
            sys.exit(1)
    
    def _clean_output(self, text: str) -> str:
        """Remove unwanted prefixes and clean output"""
        # Remove common prefixes
        prefixes = [
            "Sure, here is the Romanized version:",
            "Here is the Romanized version:",
            "Here is the romanized text:",
            "Romanized text:",
            "Output:",
            "Result:",
            "I'll convert that for you:",
            "Here's the romanized text:",
        ]
        
        for prefix in prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()
        
        # Remove quotes
        text = re.sub(r'^["\']|["\']$', '', text)
        
        # Remove any remaining system prefixes
        text = re.sub(r'^(System:|Assistant:|User:)', '', text.strip())
        
        # Ensure proper spacing
        text = re.sub(r'\s+', ' ', text)
        
        # Split by newlines and process each line
        lines = text.split('\n')
        cleaned_lines = []
        for line in lines:
            line = line.strip()
            if line and not line.startswith(('Sure', 'Here', 'Output', 'Result')):
                cleaned_lines.append(line)
        
        if cleaned_lines:
            text = ' '.join(cleaned_lines)
        else:
            text = text.strip()
        
        # Capitalize first letter of first sentence
        if text and text[0].islower():
            text = text[0].upper() + text[1:]
        
        return text.strip()
    
    def romanize(self, text: str) -> str:
        """Convert Devanagari Nepali text to Roman script"""
        if not text or not text.strip():
            return ""
        
        try:
            # Clean input text
            text = text.strip()
            
            # Invoke the model
            result = self.chain.invoke({"devanagari_text": text})
            
            # Clean up the result
            result = result.strip()
            result = self._clean_output(result)
            
            # Additional post-processing for common Nepali patterns
            result = re.sub(r'\s+', ' ', result)  # Normalize spaces
            result = re.sub(r'([a-z])([A-Z])', r'\1 \2', result)  # Add spaces at word boundaries if needed
            
            return result
            
        except Exception as e:
            print(f"✗ Error converting text: {e}", file=sys.stderr)
            print(f"  Text: {text[:50]}...", file=sys.stderr)
            return text
    
    def romanize_batch(self, texts: list) -> list:
        """Convert multiple texts to Roman script"""
        results = []
        total = len(texts)
        
        for idx, text in enumerate(texts, 1):
            if idx % 10 == 0:
                print(f"  Processing {idx}/{total}...", file=sys.stderr)
            
            result = self.romanize(text)
            results.append(result)
        
        return results


def main():
    input_file = "devnagari.txt"
    output_file = "output.txt"
    
    # Path to your local GGUF model file
    MODEL_PATH = "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf"
    TEMPERATURE = 0.0
    
    print("=" * 60, file=sys.stderr)
    print("NEPALI ROMANIZATION WITH LOCAL MISTRAL 7B", file=sys.stderr)
    print("=" * 60, file=sys.stderr)
    print(f"Model: {MODEL_PATH}", file=sys.stderr)
    print(f"Temperature: {TEMPERATURE}", file=sys.stderr)
    print("", file=sys.stderr)
    
    # Initialize the romanizer
    romanizer = NepaliRomanizerLocal(
        model_path=MODEL_PATH,
        temperature=TEMPERATURE
    )
    
    # Read input file
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]
        print(f"✓ Read {len(lines)} lines from '{input_file}'", file=sys.stderr)
        print("", file=sys.stderr)
    except FileNotFoundError:
        print(f"✗ Error: Input file '{input_file}' not found.", file=sys.stderr)
        sys.exit(1)
    
    # Process lines
    print("Converting to Roman script...", file=sys.stderr)
    results = romanizer.romanize_batch(lines)
    
    # Write output file
    with open(output_file, 'w', encoding='utf-8') as out:
        for res in results:
            out.write(res + '\n')
    
    print("", file=sys.stderr)
    print("=" * 60, file=sys.stderr)
    print(f"✓ Conversion complete!", file=sys.stderr)
    print(f"  Output written to: '{output_file}'", file=sys.stderr)
    print("=" * 60, file=sys.stderr)


if __name__ == "__main__":
    main()

NEPALI ROMANIZATION WITH LOCAL MISTRAL 7B
Model: Mistral-7B-Instruct-v0.3-Q4_K_M.gguf
Temperature: 0.0

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
✓ Model loaded successfully from 'Mistral-7B-Instruct-v0.3-Q4_K_M.gguf'
✓ Read 3 lines from 'devnagari.txt'

Converting to Roman script...

✓ Conversion complete!
  Output written to: 'output.txt'


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
from datetime import datetime
import time

In [ ]:
"""
OnlineKhabar.com - Lifestyle (and other categories) Scraper
============================================================

HTML structure confirmed by inspection:

LISTING PAGE (e.g. /lifestyle)
  div.span-3
    div.ok-sidebar-card-news.ok-card-sifaris
      div.ok-news-post.ok-post-ltr          ← each article card
        a[href]                              ← article link

ARTICLE PAGE
  div.ok-post-title-right
    h1                                       ← article title

  div.ok18-single-post-content-wrap
    p  (multiple)                            ← article body paragraphs

RELATED / "NEXT PAGE" NAVIGATION
  div.ok-section.ok-section-related
    div.ok-grid-12
      div.span-4                             ← each related article card
        a[href]                              ← link to next article

The scraper crawls the listing page to collect seed links, then follows
each article. At each article it also harvests related links (div.span-4)
and queues them — this is how it "paginates" through the site organically,
just like a reader would.
"""

import requests
from bs4 import BeautifulSoup
import time
import os
from urllib.parse import urljoin

# ─── CONFIGURATION ───────────────────────────────────────────────────────────

BASE_URL = 'https://www.onlinekhabar.com'

CATEGORIES = {
    'lifestyle':     f'{BASE_URL}/lifestyle',
    'sports':        f'{BASE_URL}/sports',
    'entertainment': f'{BASE_URL}/entertainment',
    'business':      f'{BASE_URL}/business',
    'opinion':       f'{BASE_URL}/opinion',
    'news':          f'{BASE_URL}/content/news/rastiya',
}

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/122.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'ne,en-US;q=0.9,en;q=0.8',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

MAX_ARTICLES_PER_CATEGORY = 200   # hard cap to avoid runaway crawls
PAGE_DELAY    = 0.5               # polite delay between listing page requests
ARTICLE_DELAY = 0.3               # polite delay between article requests


# ─── FETCH HELPER ────────────────────────────────────────────────────────────

def fetch(url: str):
    """
    Fetch a URL and return (BeautifulSoup, status_code).
    Returns (None, status_code) on any failure.
    """
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code == 404:
            print(f'    [404] {url}')
            return None, 404
        r.raise_for_status()
        r.encoding = 'utf-8'          # Nepali Devanagari needs explicit UTF-8
        return BeautifulSoup(r.text, 'html.parser'), r.status_code
    except requests.RequestException as e:
        print(f'    [ERROR] {url} → {e}')
        return None, 0


def normalize(href: str) -> str:
    """Ensure href is an absolute URL."""
    if href.startswith('http'):
        return href
    return urljoin(BASE_URL, href)


# ─── STEP 1 : COLLECT SEED LINKS FROM LISTING PAGE ──────────────────────────

def get_listing_links(category_url: str) -> list[str]:
    """
    Scrape the category homepage for article links using the confirmed selector:

        div.span-3
          div.ok-sidebar-card-news.ok-card-sifaris   (or similar sidebar card)
            div.ok-news-post.ok-post-ltr
              a[href]

    Also handles standard pagination (/page/2, /page/3 ...) for categories
    that use numbered pages instead of related-link crawling.
    """
    all_links = []
    seen      = set()

    soup, status = fetch(category_url)
    if not soup:
        return []

    # ── Primary selector: sidebar card layout (lifestyle homepage) ──
    # div.span-3 > div.ok-sidebar-card-news (or ok-card-sifaris) > div.ok-news-post.ok-post-ltr > a
    sidebar_sections = soup.find_all('div', class_='span-3')
    for section in sidebar_sections:
        card_wrappers = section.find_all(
            'div',
            class_=lambda c: c and ('ok-sidebar-card-news' in c or 'ok-card-sifaris' in c)
        )
        for wrapper in card_wrappers:
            posts = wrapper.find_all('div', class_=lambda c: c and 'ok-news-post' in c)
            for post in posts:
                a = post.find('a', href=True)
                if a:
                    href = normalize(a['href'])
                    if href not in seen:
                        seen.add(href)
                        all_links.append(href)

    # ── Fallback: any ok-news-post on the page (catches main feed too) ──
    if not all_links:
        posts = soup.find_all('div', class_=lambda c: c and 'ok-news-post' in c)
        for post in posts:
            a = post.find('a', href=True)
            if a:
                href = normalize(a['href'])
                if href not in seen:
                    seen.add(href)
                    all_links.append(href)

    # ── Final fallback: any internal /YEAR/MONTH/ID article URL ──
    if not all_links:
        for a in soup.find_all('a', href=True):
            href = normalize(a['href'])
            if 'onlinekhabar.com' in href and href.count('/') >= 5:
                if href not in seen:
                    seen.add(href)
                    all_links.append(href)

    print(f'  Listing page → {len(all_links)} seed links')
    return all_links


# ─── STEP 2 : EXTRACT ARTICLE CONTENT ───────────────────────────────────────

def extract_article(url: str) -> dict | None:
    """
    Extract from a single article page:
      - Site name (from <title> tag suffix or og:site_name meta)
      - Title      → div.ok-post-title-right > h1
      - Body text  → div.ok18-single-post-content-wrap > p (all paragraphs)
      - Related links → div.ok-section.ok-section-related div.span-4 > a[href]
    """
    soup, status = fetch(url)
    if not soup:
        return None

    # ── Site name ──────────────────────────────────────────────────────────
    site_name = 'OnlineKhabar'
    og_site = soup.find('meta', property='og:site_name')
    if og_site and og_site.get('content'):
        site_name = og_site['content'].strip()

    # ── Title: div.ok-post-title-right > h1 ───────────────────────────────
    title = ''
    title_div = soup.find('div', class_='ok-post-title-right')
    if title_div:
        h1 = title_div.find('h1')
        if h1:
            title = h1.get_text(strip=True)

    # Fallback title sources
    if not title:
        h1 = soup.find('h1')
        if h1:
            title = h1.get_text(strip=True)
    if not title:
        og_title = soup.find('meta', property='og:title')
        if og_title:
            title = og_title.get('content', '').strip()

    # ── Body: div.ok18-single-post-content-wrap > p ───────────────────────
    body_paragraphs = []
    content_wrap = soup.find('div', class_='ok18-single-post-content-wrap')
    if content_wrap:
        # Remove noise: ads, scripts, figures
        for noise in content_wrap.find_all(['script', 'style', 'figure', 'aside', 'ins']):
            noise.decompose()
        for p in content_wrap.find_all('p'):
            text = p.get_text(strip=True)
            if text:
                body_paragraphs.append(text)

    # Fallback body selectors if primary not found
    if not body_paragraphs:
        for fallback_sel in [
            'div.ok-single-post-content',
            'div.ok-details-post-content',
            'div.entry-content',
            'article',
        ]:
            fallback_div = soup.select_one(fallback_sel)
            if fallback_div:
                for noise in fallback_div.find_all(['script', 'style', 'figure', 'aside']):
                    noise.decompose()
                for p in fallback_div.find_all('p'):
                    text = p.get_text(strip=True)
                    if text:
                        body_paragraphs.append(text)
                if body_paragraphs:
                    break

    # ── Related links: div.ok-section.ok-section-related > div.ok-grid-12 > div.span-4 > a ──
    related_links = []
    related_section = soup.find('div', class_=lambda c: c and 'ok-section-related' in c)
    if related_section:
        grid = related_section.find('div', class_='ok-grid-12')
        if grid:
            span4_divs = grid.find_all('div', class_='span-4')
            for span in span4_divs:
                a = span.find('a', href=True)
                if a:
                    href = normalize(a['href'])
                    related_links.append(href)

    full_text = f'TITLE: {title}\n\n' + '\n\n'.join(body_paragraphs)

    return {
        'url':           url,
        'site_name':     site_name,
        'title':         title,
        'body':          body_paragraphs,
        'full_text':     full_text,
        'p_count':       len(body_paragraphs),
        'total_length':  len(full_text),
        'related_links': related_links,
    }


# ─── STEP 3 : CRAWL A CATEGORY ───────────────────────────────────────────────

def scrape_category(category_name: str, category_url: str) -> list:
    """
    Full crawl of a category:
    1. Collect seed links from listing page
    2. For each article: extract content + harvest related links
    3. Add unseen related links to the queue (BFS crawl)
    4. Respect MAX_ARTICLES_PER_CATEGORY limit
    5. Save individual .txt files + one combined file
    """
    print(f'\n{"="*70}')
    print(f'SCRAPING CATEGORY : {category_name.upper()}')
    print(f'START URL         : {category_url}')
    print(f'MAX ARTICLES      : {MAX_ARTICLES_PER_CATEGORY}')
    print(f'{"="*70}')

    os.makedirs(category_name, exist_ok=True)

    # ── Collect seed links from listing page ─────────────────────────────
    print('\n[STEP 1] Collecting seed links from listing page...')
    queue   = get_listing_links(category_url)
    visited = set(queue)

    if not queue:
        print(f'  [SKIP] No seed links found for: {category_name}')
        return []

    # ── Crawl articles + harvest related links ────────────────────────────
    print(f'\n[STEP 2] Crawling articles (max {MAX_ARTICLES_PER_CATEGORY})...')

    all_data      = []
    success_count = 0
    failed_count  = 0
    article_num   = 0

    while queue and article_num < MAX_ARTICLES_PER_CATEGORY:
        url = queue.pop(0)
        article_num += 1

        print(f'\n  [{article_num}] {url}')
        content = extract_article(url)

        if content:
            content['category'] = category_name
            all_data.append(content)
            success_count += 1

            # Queue unseen related links
            new_related = [
                l for l in content['related_links']
                if l not in visited
            ]
            for l in new_related:
                visited.add(l)
            queue.extend(new_related)

            if new_related:
                print(f'    + Queued {len(new_related)} related links (queue size: {len(queue)})')

            # Save individual article file
            fname = os.path.join(category_name, f'article_{article_num:04d}.txt')
            with open(fname, 'w', encoding='utf-8') as f:
                f.write(f'SITE     : {content["site_name"]}\n')
                f.write(f'URL      : {content["url"]}\n')
                f.write(f'CATEGORY : {category_name}\n')
                f.write(f'ARTICLE  : {article_num}\n')
                f.write(f'TITLE    : {content["title"]}\n')
                f.write(f'PARAGRAPHS: {content["p_count"]}\n')
                f.write(f'LENGTH   : {content["total_length"]} chars\n')
                f.write('='*60 + '\n\n')
                f.write(content['full_text'])

            print(f'    ✓ "{content["title"][:60]}" ({content["total_length"]} chars)')

        else:
            failed_count += 1
            print(f'    ✗ Failed to extract content')

        time.sleep(ARTICLE_DELAY)

    # ── Write combined output file ─────────────────────────────────────────
    print(f'\n[STEP 3] Writing combined file...')
    combined_file = f'{category_name}_articles.txt'

    with open(combined_file, 'w', encoding='utf-8') as f:
        f.write(f'SITE     : {BASE_URL}\n')
        f.write(f'CATEGORY : {category_name}\n')
        f.write(f'TOTAL    : {len(all_data)}\n')
        f.write(f'SUCCESS  : {success_count}\n')
        f.write(f'FAILED   : {failed_count}\n')
        f.write(f'DATE     : {time.strftime("%Y-%m-%d %H:%M:%S")}\n')
        f.write('='*80 + '\n\n')

        for i, art in enumerate(all_data, 1):
            f.write(f'─── ARTICLE {i} of {len(all_data)} ───\n')
            f.write(f'SITE     : {art["site_name"]}\n')
            f.write(f'URL      : {art["url"]}\n')
            f.write(f'TITLE    : {art["title"]}\n')
            f.write(f'CATEGORY : {art["category"]}\n')
            f.write(f'LENGTH   : {art["total_length"]} chars\n')
            f.write('-'*60 + '\n\n')
            f.write(art['full_text'])
            f.write('\n\n' + '='*80 + '\n\n')

    print(f'\n{"="*70}')
    print(f'DONE : {category_name.upper()}')
    print(f'{"="*70}')
    print(f'  ✓ Articles scraped  : {success_count}')
    print(f'  ✗ Failed            : {failed_count}')
    print(f'  ✓ Individual files  : ./{category_name}/')
    print(f'  ✓ Combined file     : {combined_file}')

    return all_data


# ─── MAIN ────────────────────────────────────────────────────────────────────

def main():
    print('='*80)
    print('ONLINEKHABAR.COM — LIFESTYLE SCRAPER (updated selectors)')
    print(f'Max {MAX_ARTICLES_PER_CATEGORY} articles per category')
    print('='*80)

    start_time     = time.time()
    total_articles = 0

    for category_name, category_url in CATEGORIES.items():
        try:
            data = scrape_category(category_name, category_url)
            total_articles += len(data)
        except Exception as e:
            import traceback
            print(f'\n[ERROR] {category_name}: {e}')
            traceback.print_exc()
            continue
        print('\nWaiting 2 seconds before next category...')
        time.sleep(2)

    elapsed = time.time() - start_time

    print(f'\n{"="*80}')
    print('ALL DONE!')
    print(f'{"="*80}')
    print(f'  Categories  : {len(CATEGORIES)}')
    print(f'  Articles    : {total_articles}')
    print(f'  Time        : {elapsed:.1f}s')
    if elapsed > 0:
        print(f'  Speed       : {total_articles / elapsed:.2f} articles/sec')
    print(f'  Finished at : {time.strftime("%Y-%m-%d %H:%M:%S")}')
    print('='*80)


if __name__ == '__main__':
    main()

ONLINEKHABAR.COM — LIFESTYLE SCRAPER (updated selectors)
Max 200 articles per category

SCRAPING CATEGORY : LIFESTYLE
START URL         : https://www.onlinekhabar.com/lifestyle
MAX ARTICLES      : 200

[STEP 1] Collecting seed links from listing page...
  Listing page → 4 seed links

[STEP 2] Crawling articles (max 200)...

  [1] https://www.onlinekhabar.com/2026/04/1902733/10-things-10-year-olds-should-know
    + Queued 4 related links (queue size: 7)
    ✓ "१० वर्षका बालबालिकाले जान्नुपर्ने १० कुरा" (3408 chars)

  [2] https://www.onlinekhabar.com/2026/03/1900844/how-to-avoid-bad-friends-7-important-signs
    + Queued 6 related links (queue size: 12)
    ✓ "नराम्रो साथीबाट कसरी जोगिने ? ७ महत्वपूर्ण संकेत" (4113 chars)

  [3] https://www.onlinekhabar.com/2026/03/1895813/350-incidents-of-child-rights-violations-during-elections
    + Queued 6 related links (queue size: 17)
    ✓ "निर्वाचनमा बालअधिकार उलंघनका ३०५ घटना" (1725 chars)

  [4] https://www.onlinekhabar.com/2026/03/1892536/se